# 04 — Data Understanding dan Audit Dataset Raw Komentar YouTube

## Judul Project

**Analisis Sentimen Komentar YouTube terhadap Isu Pelemahan Nilai Rupiah dan Dampaknya terhadap Persepsi Daya Beli Masyarakat Menggunakan Python, Jupyter Notebook, GitHub, dan Streamlit**

## Tujuan Tahap

Notebook ini bertujuan untuk melakukan **data understanding** dan **audit awal** terhadap dataset raw komentar YouTube yang telah diperoleh dari proses crawling menggunakan YouTube Data API v3.

Fokus utama tahap ini adalah memahami kualitas dataset raw sebelum masuk ke tahap cleaning, preprocessing NLP, labeling sentimen, modeling, evaluasi model, dan deployment Streamlit.

## Batasan Tahap

Tahap ini hanya mencakup:

1. Deteksi file dataset raw terbaru dari folder `data/raw/`.
2. Load dataset raw secara aman.
3. Validasi struktur kolom.
4. Audit ukuran dataset, tipe data, missing value, dan duplikasi.
5. Analisis distribusi komentar berdasarkan video, channel, dan waktu.
6. Audit statistik numerik untuk `like_count` dan `reply_count`.
7. Audit panjang teks komentar, komentar kosong, komentar terlalu pendek, dan komentar berpotensi noise.
8. Penyimpanan laporan audit agregat yang aman ke folder `reports/`.

Tahap ini **tidak melakukan** cleaning, labeling, modeling, atau pembuatan aplikasi Streamlit.

## Prinsip Keamanan Data

Dataset raw masih dapat memuat informasi sensitif seperti `author`. Oleh karena itu:

- API key tidak dibaca dan tidak ditampilkan pada notebook ini.
- Kolom `author` tidak ditampilkan secara terbuka.
- Preview data menggunakan masking pada kolom `author`.
- File raw di `data/raw/` tidak boleh dipublikasikan ke GitHub.
- File laporan audit yang disimpan ke `reports/` hanya berupa ringkasan agregat.

## Output yang Dihasilkan

Output utama notebook ini adalah:

1. Informasi file raw terbaru yang terdeteksi.
2. DataFrame `df_raw` yang berisi dataset raw untuk kebutuhan audit.
3. Preview aman dengan masking kolom `author`.
4. Tabel validasi kolom wajib dan kolom opsional.
5. Ringkasan jumlah baris, jumlah kolom, tipe data, dan unique value.
6. Ringkasan missing value.
7. Ringkasan duplikasi berdasarkan `comment_id` dan `text_original`.
8. Distribusi komentar per video dan per channel.
9. Rentang tanggal komentar dan jumlah komentar per hari/bulan.
10. Statistik `like_count` dan `reply_count`.
11. Statistik panjang komentar.
12. Ringkasan komentar kosong, terlalu pendek, dan berpotensi noise.
13. Ringkasan kelayakan dataset sebelum tahap cleaning.
14. File laporan audit aman di folder `reports/`.

## 1. Import Library

Bagian ini melakukan import library dasar untuk membaca dataset dan melakukan audit. Notebook ini tidak memanggil YouTube API dan tidak membaca file `.env`.

In [10]:
# ============================================================
# 04 - Data Understanding dan Audit Dataset Raw Komentar YouTube
# Bagian 1: Import Library
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

print("Library berhasil di-import.")

Library berhasil di-import.


## 2. Deteksi Project Root dan Folder Kerja

Project root adalah folder utama project:

`youtube-rupiah-sentiment-analysis`

Deteksi project root diperlukan agar notebook tetap dapat berjalan baik saat dijalankan dari folder `notebooks/` maupun dari folder utama project.

In [11]:
# ============================================================
# Bagian 2: Deteksi Project Root dan Folder Kerja
# ============================================================

def find_project_root(start_path: Path) -> Path:
    '''
    Mendeteksi root project berdasarkan keberadaan folder:
    - notebooks
    - data
    - .git

    Fungsi ini menjaga notebook tetap portable.
    '''
    start_path = start_path.resolve()

    for candidate in [start_path] + list(start_path.parents):
        has_notebooks = (candidate / "notebooks").exists()
        has_data = (candidate / "data").exists()
        has_git = (candidate / ".git").exists()

        if (has_notebooks and has_data) or has_git:
            return candidate

    raise FileNotFoundError(
        "Project root tidak ditemukan. "
        "Pastikan notebook berada di dalam folder project youtube-rupiah-sentiment-analysis."
    )

CURRENT_PATH = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_PATH)

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Membuat folder reports jika belum tersedia.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Current Path :", CURRENT_PATH)
print("Project Root :", PROJECT_ROOT)
print("Raw Data Dir :", RAW_DATA_DIR)
print("Reports Dir  :", REPORTS_DIR)

Current Path : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\notebooks
Project Root : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis
Raw Data Dir : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\data\raw
Reports Dir  : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis\reports


## 3. Deteksi File Dataset Raw Terbaru

Dataset raw hasil crawling diharapkan berada pada folder:

`data/raw/`

Notebook akan mencari file dataset terbaru berdasarkan waktu modifikasi file. Format yang didukung:

- `.csv`
- `.xlsx`
- `.xls`
- `.json`
- `.jsonl`
- `.parquet`

Output bagian ini hanya metadata file, bukan isi komentar.

In [12]:
# ============================================================
# Bagian 3: Deteksi File Dataset Raw Terbaru
# ============================================================

SUPPORTED_EXTENSIONS = {".csv", ".xlsx", ".xls", ".json", ".jsonl", ".parquet"}

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder data raw tidak ditemukan: {RAW_DATA_DIR}\n"
        "Pastikan folder data/raw/ sudah dibuat dan dataset hasil crawling sudah tersimpan di dalamnya."
    )

raw_files = [
    file_path
    for file_path in RAW_DATA_DIR.iterdir()
    if (
        file_path.is_file()
        and file_path.suffix.lower() in SUPPORTED_EXTENSIONS
        and not file_path.name.startswith("~$")
    )
]

if len(raw_files) == 0:
    raise FileNotFoundError(
        f"Tidak ditemukan file dataset raw di folder: {RAW_DATA_DIR}\n"
        "Pastikan file hasil crawling dari notebook 03 sudah tersimpan di folder data/raw/."
    )

raw_files_sorted = sorted(
    raw_files,
    key=lambda file_path: file_path.stat().st_mtime,
    reverse=True
)

raw_file_inventory = pd.DataFrame([
    {
        "no": idx + 1,
        "file_name": file_path.name,
        "extension": file_path.suffix.lower(),
        "size_mb": round(file_path.stat().st_size / (1024 * 1024), 4),
        "modified_at": datetime.fromtimestamp(file_path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "relative_path": str(file_path.relative_to(PROJECT_ROOT))
    }
    for idx, file_path in enumerate(raw_files_sorted)
])

LATEST_RAW_FILE = raw_files_sorted[0]

print("Jumlah file dataset raw terdeteksi:", len(raw_files_sorted))
print("File dataset raw terbaru:", LATEST_RAW_FILE.name)
print("Lokasi relatif:", LATEST_RAW_FILE.relative_to(PROJECT_ROOT))

display(raw_file_inventory)

Jumlah file dataset raw terdeteksi: 2
File dataset raw terbaru: youtube_comments_raw_20260529_141537.csv
Lokasi relatif: data\raw\youtube_comments_raw_20260529_141537.csv


,no,file_name,extension,size_mb,modified_at,relative_path
0,1,youtube_comments_raw_20260529_141537.csv,.csv,1.3799,2026-05-29 14:16:48,data\raw\youtube_comments_raw_20260529_141537.csv
1,2,youtube_video_candidates_20260529_141537.csv,.csv,0.0040,2026-05-29 14:15:37,data\raw\youtube_video_candidates_20260529_141537.csv


## 4. Load Dataset Raw Secara Aman

Bagian ini membaca file dataset raw terbaru ke dalam DataFrame bernama `df_raw`.

Penting: error `NameError: name 'df_raw' is not defined` biasanya terjadi karena bagian load dataset belum dijalankan, sementara cell audit kolom sudah dijalankan. Oleh karena itu, pastikan cell ini berhasil dijalankan sebelum lanjut ke bagian validasi struktur kolom.

In [13]:
# ============================================================
# Bagian 4: Load Dataset Raw Secara Aman
# ============================================================

def load_dataset(file_path: Path) -> pd.DataFrame:
    '''
    Membaca dataset berdasarkan ekstensi file.
    '''
    extension = file_path.suffix.lower()

    if extension == ".csv":
        try:
            return pd.read_csv(file_path, encoding="utf-8-sig")
        except UnicodeDecodeError:
            return pd.read_csv(file_path, encoding="utf-8")

    if extension in [".xlsx", ".xls"]:
        return pd.read_excel(file_path)

    if extension == ".json":
        return pd.read_json(file_path)

    if extension == ".jsonl":
        return pd.read_json(file_path, lines=True)

    if extension == ".parquet":
        return pd.read_parquet(file_path)

    raise ValueError(f"Format file belum didukung: {extension}")

df_raw = load_dataset(LATEST_RAW_FILE)

print("Dataset raw berhasil dibaca.")
print("Nama file    :", LATEST_RAW_FILE.name)
print("Jumlah baris :", df_raw.shape[0])
print("Jumlah kolom :", df_raw.shape[1])

Dataset raw berhasil dibaca.
Nama file    : youtube_comments_raw_20260529_141537.csv
Jumlah baris : 3742
Jumlah kolom : 13


## 5. Preview Aman Dataset Raw

Preview dilakukan secara terbatas untuk memastikan data berhasil dibaca. Kolom `author` dimasking agar identitas penulis komentar tidak tampil secara langsung.

In [14]:
# ============================================================
# Bagian 5: Preview Aman Dataset Raw
# ============================================================

def mask_author(author):
    '''
    Masking sederhana untuk menyembunyikan identitas author.
    '''
    if pd.isna(author):
        return np.nan

    author = str(author)

    if len(author) <= 2:
        return "*" * len(author)

    return author[0] + "*" * (len(author) - 2) + author[-1]

df_preview = df_raw.copy()

if "author" in df_preview.columns:
    df_preview["author"] = df_preview["author"].apply(mask_author)

display(df_preview.head(5))

,video_id,video_title,video_url,channel_title,video_published_at,comment_id,author,published_at,comment_updated_at,text_original,like_count,reply_count,crawl_timestamp
0,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),https://www.youtube.com/watch?v=sB7fSPbYFJg,Kok Bisa?,2015-08-23T13:44:50Z,Ugzo_OAorVrtNPFL3cF4AaABAg,@*******y,2026-05-24 01:37:15+00:00,2026-05-24T01:37:15Z,"gue kira di upload 10 jam yg lalu,ternyata 10 tahun yg lalu😂",560,11,2026-05-29 07:15:52 UTC
1,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),https://www.youtube.com/watch?v=sB7fSPbYFJg,Kok Bisa?,2015-08-23T13:44:50Z,Ugx2WJ9HFPUNRPaOxON4AaABAg,@*******l,2026-05-22 13:31:00+00:00,2026-05-26T12:17:30Z,"Timing nya gabisa lebih pas dari ini😭🙏\nNyentuh Rp.25.000,00 Masih Aman Indonesia😎👌",570,12,2026-05-29 07:15:52 UTC
2,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),https://www.youtube.com/watch?v=sB7fSPbYFJg,Kok Bisa?,2015-08-23T13:44:50Z,UgwyjT6JiPsbjsMoUqR4AaABAg,@********i,2026-05-24 09:42:48+00:00,2026-05-24T09:42:48Z,Algoritma YouTube cukup mengerikan 🥀,271,3,2026-05-29 07:15:52 UTC
3,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),https://www.youtube.com/watch?v=sB7fSPbYFJg,Kok Bisa?,2015-08-23T13:44:50Z,UgwN7TGK6eU5papwnNZ4AaABAg,@*******8,2026-05-21 06:57:48+00:00,2026-05-21T06:57:48Z,dyeem 10 year ago lewat beranda kuhh pas bgett;(((,321,2,2026-05-29 07:15:52 UTC
4,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),https://www.youtube.com/watch?v=sB7fSPbYFJg,Kok Bisa?,2015-08-23T13:44:50Z,UgzSaWu-A-vsRbjSc6B4AaABAg,@***************r,2026-05-23 17:47:36+00:00,2026-05-23T17:47:36Z,Algoritma YouTube lebih mengerikan daripada Pria Solo☠️,92,1,2026-05-29 07:15:52 UTC


## 6. Validasi Struktur Kolom Dataset

Validasi struktur kolom dilakukan untuk memastikan dataset raw memiliki kolom minimal yang dibutuhkan untuk analisis sentimen komentar YouTube.

Kolom wajib:

1. `video_id`
2. `video_title`
3. `comment_id`
4. `author`
5. `published_at`
6. `text_original`
7. `like_count`

Kolom opsional:

1. `video_url`
2. `channel_title`
3. `video_published_at`
4. `comment_updated_at`
5. `reply_count`
6. `crawl_timestamp`

In [15]:
# ============================================================
# Bagian 6: Validasi Struktur Kolom Dataset
# ============================================================

required_columns = [
    "video_id",
    "video_title",
    "comment_id",
    "author",
    "published_at",
    "text_original",
    "like_count"
]

optional_columns = [
    "video_url",
    "channel_title",
    "video_published_at",
    "comment_updated_at",
    "reply_count",
    "crawl_timestamp"
]

available_columns = list(df_raw.columns)

missing_required_columns = [
    col for col in required_columns
    if col not in available_columns
]

column_validation_df = pd.DataFrame({
    "column_name": required_columns + optional_columns,
    "column_category": ["required"] * len(required_columns) + ["optional"] * len(optional_columns),
    "is_available": [
        col in available_columns
        for col in required_columns + optional_columns
    ]
})

print("Jumlah kolom tersedia:", len(available_columns))
print("Kolom tersedia:")
print(available_columns)

print("\nJumlah kolom wajib yang belum tersedia:", len(missing_required_columns))

if len(missing_required_columns) > 0:
    print("Kolom wajib yang belum tersedia:", missing_required_columns)
else:
    print("Semua kolom wajib tersedia.")

display(column_validation_df)

Jumlah kolom tersedia: 13
Kolom tersedia:
['video_id', 'video_title', 'video_url', 'channel_title', 'video_published_at', 'comment_id', 'author', 'published_at', 'comment_updated_at', 'text_original', 'like_count', 'reply_count', 'crawl_timestamp']

Jumlah kolom wajib yang belum tersedia: 0
Semua kolom wajib tersedia.


,column_name,column_category,is_available
0,video_id,required,True
1,video_title,required,True
2,comment_id,required,True
3,author,required,True
4,published_at,required,True
5,text_original,required,True
6,like_count,required,True
7,video_url,optional,True
8,channel_title,optional,True
9,video_published_at,optional,True


## 7. Jumlah Baris, Jumlah Kolom, Tipe Data, dan Unique Value

Bagian ini melakukan audit struktur dasar dataset, termasuk ukuran dataset, tipe data, jumlah nilai non-null, jumlah null, dan jumlah nilai unik pada setiap kolom.

In [16]:
# ============================================================
# Bagian 7: Jumlah Baris, Jumlah Kolom, Tipe Data, dan Unique Value
# ============================================================

dataset_shape_summary = pd.DataFrame({
    "metric": ["jumlah_baris", "jumlah_kolom"],
    "value": [df_raw.shape[0], df_raw.shape[1]]
})

dtype_summary_df = pd.DataFrame({
    "column_name": df_raw.columns,
    "data_type": [str(dtype) for dtype in df_raw.dtypes],
    "non_null_count": [df_raw[col].notna().sum() for col in df_raw.columns],
    "null_count": [df_raw[col].isna().sum() for col in df_raw.columns],
    "null_percentage": [round(df_raw[col].isna().mean() * 100, 2) for col in df_raw.columns],
    "unique_count": [df_raw[col].nunique(dropna=True) for col in df_raw.columns]
})

display(dataset_shape_summary)
display(dtype_summary_df)

,metric,value
0,jumlah_baris,3742
1,jumlah_kolom,13


,column_name,data_type,non_null_count,null_count,null_percentage,unique_count
0,video_id,str,3742,0,0.0,17
1,video_title,str,3742,0,0.0,17
2,video_url,str,3742,0,0.0,17
3,channel_title,str,3742,0,0.0,12
4,video_published_at,str,3742,0,0.0,17
5,comment_id,str,3742,0,0.0,3742
6,author,str,3742,0,0.0,3482
7,published_at,str,3742,0,0.0,3729
8,comment_updated_at,str,3742,0,0.0,3728
9,text_original,str,3742,0,0.0,3699


## 8. Audit Missing Value

Missing value perlu diperiksa untuk memahami apakah terdapat kolom penting yang kosong, terutama `comment_id`, `published_at`, dan `text_original`.

In [17]:
# ============================================================
# Bagian 8: Audit Missing Value
# ============================================================

missing_value_df = pd.DataFrame({
    "column_name": df_raw.columns,
    "missing_count": [df_raw[col].isna().sum() for col in df_raw.columns],
    "missing_percentage": [round(df_raw[col].isna().mean() * 100, 2) for col in df_raw.columns]
}).sort_values(by="missing_count", ascending=False)

display(missing_value_df)

,column_name,missing_count,missing_percentage
0,video_id,0,0.0
1,video_title,0,0.0
2,video_url,0,0.0
3,channel_title,0,0.0
4,video_published_at,0,0.0
5,comment_id,0,0.0
6,author,0,0.0
7,published_at,0,0.0
8,comment_updated_at,0,0.0
9,text_original,0,0.0


## 9. Audit Duplikasi Data

Duplikasi diperiksa berdasarkan dua pendekatan:

1. Duplikasi `comment_id`, karena setiap komentar YouTube idealnya memiliki ID unik.
2. Duplikasi `text_original`, karena komentar dengan isi yang sama dapat mengindikasikan spam, komentar berulang, atau noise.

Notebook hanya menampilkan ringkasan agregat, bukan isi komentar mentah.

In [18]:
# ============================================================
# Bagian 9: Audit Duplikasi Data
# ============================================================

duplicate_comment_id_count = np.nan
duplicate_comment_id_percentage = np.nan

if "comment_id" in df_raw.columns:
    duplicate_comment_id_count = int(df_raw.duplicated(subset=["comment_id"]).sum())
    duplicate_comment_id_percentage = round((duplicate_comment_id_count / len(df_raw)) * 100, 2) if len(df_raw) > 0 else 0

duplicate_text_count = np.nan
duplicate_text_percentage = np.nan

if "text_original" in df_raw.columns:
    duplicate_text_count = int(df_raw.duplicated(subset=["text_original"]).sum())
    duplicate_text_percentage = round((duplicate_text_count / len(df_raw)) * 100, 2) if len(df_raw) > 0 else 0

duplicate_summary_df = pd.DataFrame([
    {
        "duplicate_basis": "comment_id",
        "duplicate_count": duplicate_comment_id_count,
        "duplicate_percentage": duplicate_comment_id_percentage
    },
    {
        "duplicate_basis": "text_original",
        "duplicate_count": duplicate_text_count,
        "duplicate_percentage": duplicate_text_percentage
    }
])

display(duplicate_summary_df)

,duplicate_basis,duplicate_count,duplicate_percentage
0,comment_id,0,0.00
1,text_original,43,1.15


## 10. Konversi Kolom Tanggal untuk Audit

Bagian ini membuat salinan kerja bernama `df_audit`. Kolom tanggal dikonversi ke format datetime untuk menghitung rentang tanggal dan distribusi komentar berdasarkan waktu.

Dataset raw asli `df_raw` tidak dimodifikasi.

In [19]:
# ============================================================
# Bagian 10: Konversi Kolom Tanggal untuk Audit
# ============================================================

df_audit = df_raw.copy()

date_columns = [
    "published_at",
    "video_published_at",
    "comment_updated_at",
    "crawl_timestamp"
]

date_conversion_rows = []

for col in date_columns:
    if col in df_audit.columns:
        converted_col = f"{col}_dt"
        df_audit[converted_col] = pd.to_datetime(df_audit[col], errors="coerce", utc=True)

        date_conversion_rows.append({
            "column_name": col,
            "converted_column": converted_col,
            "valid_datetime_count": int(df_audit[converted_col].notna().sum()),
            "invalid_or_missing_count": int(df_audit[converted_col].isna().sum()),
            "valid_datetime_percentage": round(df_audit[converted_col].notna().mean() * 100, 2)
        })

date_conversion_summary_df = pd.DataFrame(date_conversion_rows)

display(date_conversion_summary_df)

,column_name,converted_column,valid_datetime_count,invalid_or_missing_count,valid_datetime_percentage
0,published_at,published_at_dt,3742,0,100.0
1,video_published_at,video_published_at_dt,3742,0,100.0
2,comment_updated_at,comment_updated_at_dt,3742,0,100.0
3,crawl_timestamp,crawl_timestamp_dt,3742,0,100.0


## 11. Rentang Tanggal Komentar

Rentang tanggal komentar digunakan untuk memahami periode data yang tercakup dalam dataset.

In [20]:
# ============================================================
# Bagian 11: Rentang Tanggal Komentar
# ============================================================

if "published_at_dt" in df_audit.columns:
    valid_comment_dates = df_audit["published_at_dt"].dropna()

    if len(valid_comment_dates) > 0:
        date_range_summary_df = pd.DataFrame([
            {
                "metric": "tanggal_komentar_terawal",
                "value": valid_comment_dates.min().strftime("%Y-%m-%d %H:%M:%S %Z")
            },
            {
                "metric": "tanggal_komentar_terbaru",
                "value": valid_comment_dates.max().strftime("%Y-%m-%d %H:%M:%S %Z")
            },
            {
                "metric": "jumlah_hari_cakupan_data",
                "value": int((valid_comment_dates.max() - valid_comment_dates.min()).days)
            },
            {
                "metric": "jumlah_komentar_dengan_tanggal_valid",
                "value": int(len(valid_comment_dates))
            }
        ])
    else:
        date_range_summary_df = pd.DataFrame([
            {"metric": "status", "value": "Tidak ada tanggal komentar valid."}
        ])
else:
    date_range_summary_df = pd.DataFrame([
        {"metric": "status", "value": "Kolom published_at tidak tersedia."}
    ])

display(date_range_summary_df)

,metric,value
0,tanggal_komentar_terawal,2015-08-23 14:36:01 UTC
1,tanggal_komentar_terbaru,2026-05-29 07:14:24 UTC
2,jumlah_hari_cakupan_data,3931
3,jumlah_komentar_dengan_tanggal_valid,3742


## 12. Distribusi Komentar per Video

Distribusi komentar per video menunjukkan apakah komentar terkonsentrasi pada video tertentu atau relatif tersebar pada beberapa video.

In [21]:
# ============================================================
# Bagian 12: Distribusi Komentar per Video
# ============================================================

if "video_id" in df_raw.columns:
    group_columns = ["video_id"]

    if "video_title" in df_raw.columns:
        group_columns.append("video_title")

    if "channel_title" in df_raw.columns:
        group_columns.append("channel_title")

    comments_per_video_df = (
        df_raw
        .groupby(group_columns, dropna=False)
        .size()
        .reset_index(name="comment_count")
        .sort_values(by="comment_count", ascending=False)
    )

    comments_per_video_df["comment_percentage"] = round(
        comments_per_video_df["comment_count"] / len(df_raw) * 100, 2
    )
else:
    comments_per_video_df = pd.DataFrame([
        {"status": "Kolom video_id tidak tersedia."}
    ])

display(comments_per_video_df.head(20))

,video_id,video_title,channel_title,comment_count,comment_percentage
3,DVJZCxeBfMA,DAMPAK RUPIAH MELEMAH 1 JADI GILA ❌ SEMUA WARGA JADI GILA ✅🫵🏼😭,poin penting,500,13.36
13,sGgYsNt2ick,Akar Masalah Pelemahan Rupiah,Ngomongin Uang,500,13.36
14,u72ts1RA2oQ,PENJELASAN: Kenapa Harga Dolar Naik & Rupiah Melemah?,Ngomongin Uang,500,13.36
11,nF0McI5sqBA,Sejarah Rupiah: Kenapa Nilainya Selalu Turun? KATA BUZZER STRATEGI PEMERINTAH! |Learning By Googling,Sepulang Sekolah,500,13.36
12,sB7fSPbYFJg,Kenapa Rupiah Melemah? (Explained),Kok Bisa?,500,13.36
0,--_k_ACPYTk,"Rupiah Makin Lemah, Daya Beli Mulai Goyah? | Prime Plus",CNN Indonesia,474,12.67
4,GXLA2I3bifE,"Rupiah Melemah Tajam Rp17 800, Menkeu Purbaya: Tak Masuk Akal, Fundamental Ekonomi Kita Bagus!",KOMPASTV JAWA TIMUR,260,6.95
5,LHPZRgUL6tg,"Rupiah Melemah, Apa yang Terjadi jika Tembus Rp 25.000 Per Dollar AS?",Kompas.com,208,5.56
8,Wcq5TnmXQXU,"Rupiah Tembus Level Terlemah, Dampak Merembet ke Harga Kebutuhan Pokok | BEDAH DATA",Nusantara TV,118,3.15
15,vvLBbx0q9F4,Roy Mandey: Masyarakat Cenderung Tidak Lagi Belanja Bulanan | Prime Plus Part 1,CNN Indonesia,94,2.51


## 13. Distribusi Komentar per Channel

Distribusi komentar per channel membantu melihat sumber video yang paling banyak menyumbang komentar dalam dataset.

In [22]:
# ============================================================
# Bagian 13: Distribusi Komentar per Channel
# ============================================================

if "channel_title" in df_raw.columns:
    comments_per_channel_df = (
        df_raw
        .groupby("channel_title", dropna=False)
        .size()
        .reset_index(name="comment_count")
        .sort_values(by="comment_count", ascending=False)
    )

    comments_per_channel_df["comment_percentage"] = round(
        comments_per_channel_df["comment_count"] / len(df_raw) * 100, 2
    )
else:
    comments_per_channel_df = pd.DataFrame([
        {"status": "Kolom channel_title tidak tersedia."}
    ])

display(comments_per_channel_df.head(20))

,channel_title,comment_count,comment_percentage
6,Ngomongin Uang,1000,26.72
0,CNN Indonesia,573,15.31
4,Kok Bisa?,500,13.36
10,poin penting,500,13.36
8,Sepulang Sekolah,500,13.36
3,KOMPASTV JAWA TIMUR,260,6.95
5,Kompas.com,247,6.60
7,Nusantara TV,136,3.63
11,tvOneNews,15,0.40
9,kumparan,9,0.24


## 14. Jumlah Komentar per Hari dan per Bulan

Distribusi waktu membantu memahami pola komentar berdasarkan tanggal publikasi komentar.

In [23]:
# ============================================================
# Bagian 14: Jumlah Komentar per Hari dan per Bulan
# ============================================================

if "published_at_dt" in df_audit.columns and df_audit["published_at_dt"].notna().sum() > 0:
    df_audit["comment_date"] = df_audit["published_at_dt"].dt.date
    df_audit["comment_month"] = df_audit["published_at_dt"].dt.to_period("M").astype(str)

    comments_per_day_df = (
        df_audit
        .groupby("comment_date", dropna=False)
        .size()
        .reset_index(name="comment_count")
        .sort_values(by="comment_date")
    )

    comments_per_month_df = (
        df_audit
        .groupby("comment_month", dropna=False)
        .size()
        .reset_index(name="comment_count")
        .sort_values(by="comment_month")
    )
else:
    comments_per_day_df = pd.DataFrame([
        {"status": "Tanggal komentar tidak tersedia atau tidak valid."}
    ])
    comments_per_month_df = pd.DataFrame([
        {"status": "Tanggal komentar tidak tersedia atau tidak valid."}
    ])

display(comments_per_day_df.head(20))
display(comments_per_month_df)

,comment_date,comment_count
0,2015-08-23,1
1,2015-08-24,12
2,2015-08-25,9
3,2015-08-26,2
4,2015-08-27,1
5,2015-09-01,1
6,2015-09-02,1
7,2016-03-07,1
8,2016-03-09,1
9,2016-03-15,1


,comment_month,comment_count
0,2015-08,25
1,2015-09,2
2,2016-03,3
3,2016-04,1
4,2016-05,1
...,...,...
97,2025-10,1
98,2026-01,18
99,2026-03,2
100,2026-04,1


## 15. Statistik `like_count` dan `reply_count`

Bagian ini menghitung statistik deskriptif untuk kolom numerik `like_count` dan `reply_count`.

In [24]:
# ============================================================
# Bagian 15: Statistik like_count dan reply_count
# ============================================================

numeric_columns = []

for col in ["like_count", "reply_count"]:
    if col in df_audit.columns:
        df_audit[col] = pd.to_numeric(df_audit[col], errors="coerce")
        numeric_columns.append(col)

if len(numeric_columns) > 0:
    numeric_stats_df = (
        df_audit[numeric_columns]
        .describe()
        .T
        .reset_index()
        .rename(columns={"index": "column_name"})
    )

    numeric_stats_df["missing_count"] = [
        int(df_audit[col].isna().sum()) for col in numeric_columns
    ]
else:
    numeric_stats_df = pd.DataFrame([
        {"status": "Kolom like_count dan reply_count tidak tersedia."}
    ])

display(numeric_stats_df)

,column_name,count,mean,std,min,25%,50%,75%,max,missing_count
0,like_count,3742.0,7.419562,83.546057,0.0,0.0,0.0,1.0,3661.0,0
1,reply_count,3742.0,0.755478,5.347663,0.0,0.0,0.0,0.0,148.0,0


## 16. Audit Panjang Teks Komentar

Bagian ini menghitung panjang teks komentar berdasarkan jumlah karakter dan jumlah kata. Informasi ini penting untuk menilai kesiapan dataset sebelum preprocessing NLP.

In [25]:
# ============================================================
# Bagian 16: Audit Panjang Teks Komentar
# ============================================================

if "text_original" not in df_audit.columns:
    raise KeyError("Kolom text_original tidak tersedia. Audit teks tidak dapat dilakukan.")

df_audit["text_original_safe"] = df_audit["text_original"].fillna("").astype(str)
df_audit["text_stripped"] = df_audit["text_original_safe"].str.strip()
df_audit["text_length_char"] = df_audit["text_stripped"].str.len()
df_audit["text_word_count"] = df_audit["text_stripped"].apply(lambda x: len(x.split()) if x else 0)

text_length_stats_df = pd.DataFrame([
    {
        "metric": "jumlah_komentar",
        "value": int(len(df_audit))
    },
    {
        "metric": "rata_rata_panjang_karakter",
        "value": round(df_audit["text_length_char"].mean(), 2)
    },
    {
        "metric": "median_panjang_karakter",
        "value": round(df_audit["text_length_char"].median(), 2)
    },
    {
        "metric": "min_panjang_karakter",
        "value": int(df_audit["text_length_char"].min())
    },
    {
        "metric": "max_panjang_karakter",
        "value": int(df_audit["text_length_char"].max())
    },
    {
        "metric": "rata_rata_jumlah_kata",
        "value": round(df_audit["text_word_count"].mean(), 2)
    },
    {
        "metric": "median_jumlah_kata",
        "value": round(df_audit["text_word_count"].median(), 2)
    }
])

display(text_length_stats_df)

,metric,value
0,jumlah_komentar,3742.00
1,rata_rata_panjang_karakter,109.02
2,median_panjang_karakter,65.00
3,min_panjang_karakter,1.00
4,max_panjang_karakter,2457.00
5,rata_rata_jumlah_kata,16.78
6,median_jumlah_kata,11.00


## 17. Audit Komentar Kosong, Terlalu Pendek, dan Berpotensi Noise

Komentar berpotensi noise pada tahap audit awal didefinisikan menggunakan indikator sederhana:

1. Komentar kosong.
2. Komentar terlalu pendek.
3. Komentar hanya berisi simbol/tanda baca.
4. Komentar mengandung URL.
5. Komentar memiliki pola karakter berulang berlebihan.
6. Komentar duplikat berdasarkan teks asli.

Catatan: Pada tahap ini indikator noise hanya bersifat deteksi awal. Keputusan final pembersihan data dilakukan pada tahap cleaning.

In [32]:
# ============================================================
# Audit Komentar Kosong, Terlalu Pendek, dan Berpotensi Noise
# Versi final: aman dari ArrowInvalid dan konsisten untuk ringkasan audit
# ============================================================

import re
import numpy as np
import pandas as pd

# Membuat salinan untuk proses audit agar df_raw tetap utuh
df_audit = df_raw.copy()

# Validasi kolom teks utama
if "text_original" not in df_audit.columns:
    raise KeyError("Kolom 'text_original' tidak ditemukan pada dataset raw.")

# Konversi teks ke object/string Python biasa agar aman untuk regex Python
df_audit["text_original_safe"] = (
    df_audit["text_original"]
    .fillna("")
    .astype(object)
    .astype(str)
)

# Menghapus spasi di awal dan akhir teks
df_audit["text_stripped"] = df_audit["text_original_safe"].apply(lambda x: x.strip())

# Panjang karakter komentar
df_audit["text_length"] = df_audit["text_stripped"].apply(len)

# Jumlah kata komentar
df_audit["word_count"] = df_audit["text_stripped"].apply(
    lambda x: len(x.split()) if x.strip() != "" else 0
)

# Komentar kosong
df_audit["is_empty_comment"] = df_audit["text_stripped"].eq("")

# Komentar terlalu pendek
# Definisi awal:
# - panjang karakter <= 3, atau
# - jumlah kata <= 1
df_audit["is_too_short"] = (
    (df_audit["text_length"] <= 3) |
    (df_audit["word_count"] <= 1)
)

# Komentar hanya simbol/tanda baca
df_audit["is_symbol_only"] = df_audit["text_stripped"].apply(
    lambda x: bool(x) and len(re.sub(r"[\W_]+", "", x, flags=re.UNICODE)) == 0
)

# Komentar mengandung URL
df_audit["has_url"] = df_audit["text_stripped"].apply(
    lambda x: bool(re.search(r"(https?://|www\.)", x, flags=re.IGNORECASE))
)

# Komentar dengan karakter berulang berlebihan
# Menggunakan Python re agar aman dari error ArrowInvalid pada regex backreference \1
df_audit["has_excessive_repetition"] = df_audit["text_stripped"].apply(
    lambda x: bool(re.search(r"(.)\1{4,}", x))
)

# Komentar pendek berupa noise umum
noise_keywords_pattern = r"^(wkwk+|haha+|hehe+|ok+|oke+|sip+|mantap+|gas+|up+|lol+)$"

df_audit["is_short_noise_keyword"] = df_audit["text_stripped"].str.lower().apply(
    lambda x: bool(re.search(noise_keywords_pattern, x.strip()))
)

# Gabungan indikator komentar berpotensi noise
df_audit["is_potential_noise"] = (
    df_audit["is_empty_comment"] |
    df_audit["is_too_short"] |
    df_audit["is_symbol_only"] |
    df_audit["has_url"] |
    df_audit["has_excessive_repetition"] |
    df_audit["is_short_noise_keyword"]
)

# Ringkasan audit noise dengan format kolom yang konsisten
noise_summary_df = pd.DataFrame({
    "indicator": [
        "empty_comment",
        "too_short",
        "symbol_only",
        "contains_url",
        "excessive_repetition",
        "short_noise_keyword",
        "total_potential_noise"
    ],
    "description": [
        "Komentar kosong",
        "Komentar terlalu pendek",
        "Komentar hanya simbol/tanda baca",
        "Komentar mengandung URL",
        "Komentar dengan karakter berulang berlebihan",
        "Komentar pendek berupa noise umum",
        "Total komentar berpotensi noise"
    ],
    "count": [
        int(df_audit["is_empty_comment"].sum()),
        int(df_audit["is_too_short"].sum()),
        int(df_audit["is_symbol_only"].sum()),
        int(df_audit["has_url"].sum()),
        int(df_audit["has_excessive_repetition"].sum()),
        int(df_audit["is_short_noise_keyword"].sum()),
        int(df_audit["is_potential_noise"].sum())
    ]
})

noise_summary_df["percentage"] = (
    noise_summary_df["count"] / len(df_audit) * 100
).round(2)

display(noise_summary_df)

,indicator,description,count,percentage
0,empty_comment,Komentar kosong,0,0.00
1,too_short,Komentar terlalu pendek,77,2.06
2,symbol_only,Komentar hanya simbol/tanda baca,27,0.72
3,contains_url,Komentar mengandung URL,0,0.00
4,excessive_repetition,Komentar dengan karakter berulang berlebihan,144,3.85
5,short_noise_keyword,Komentar pendek berupa noise umum,0,0.00
6,total_potential_noise,Total komentar berpotensi noise,213,5.69


In [33]:
# ============================================================
# Sampel Aman Komentar Berpotensi Noise
# ============================================================

sample_columns = [
    "comment_id",
    "video_title",
    "text_original_safe",
    "text_length",
    "word_count",
    "is_empty_comment",
    "is_too_short",
    "is_symbol_only",
    "has_url",
    "has_excessive_repetition",
    "is_short_noise_keyword",
    "is_potential_noise"
]

available_sample_columns = [
    col for col in sample_columns
    if col in df_audit.columns
]

noise_sample_safe = (
    df_audit.loc[df_audit["is_potential_noise"], available_sample_columns]
    .head(10)
    .copy()
)

# Membatasi tampilan teks agar tidak terlalu panjang
if "text_original_safe" in noise_sample_safe.columns:
    noise_sample_safe["text_original_safe"] = noise_sample_safe["text_original_safe"].apply(
        lambda x: x[:120] + "..." if len(str(x)) > 120 else x
    )

display(noise_sample_safe)

,comment_id,video_title,text_original_safe,text_length,word_count,is_empty_comment,is_too_short,is_symbol_only,has_url,has_excessive_repetition,is_short_noise_keyword,is_potential_noise
71,UgyeL8ANBTvUWcEFzvR4AaABAg,Kenapa Rupiah Melemah? (Explained),Video 10 tahun lalu tapi relate bgt di tahun ini 😂😂😂😂😂😂,55,11,False,False,False,False,True,False,True
121,UgxiwSSLEz46cNiRXvt4AaABAg,Kenapa Rupiah Melemah? (Explained),2019?,5,1,False,True,False,False,False,False,True
206,Ugwzv56mfHayPFiLTyJ4AaABAg,Kenapa Rupiah Melemah? (Explained),"Makanyaaaaaa rakyat indo belajar investasi giiih , masih dikit bgt investor org indo",84,13,False,False,False,False,True,False,True
226,Ugz0VyV_iXOGHbIClQF4AaABAg,Kenapa Rupiah Melemah? (Explained),HEYYYYY SOOO IT'S GETTING WORST AFTER 10 YEARS LMAOOO 🥳💥,56,10,False,False,False,False,True,False,True
235,UgzfLjgDFx4jnvEMZ114AaABAg,Kenapa Rupiah Melemah? (Explained),"Ku kira baru upload, ternyata.......",36,5,False,False,False,False,True,False,True
278,UgwiixAYs-It9CVD1NZ4AaABAg,Kenapa Rupiah Melemah? (Explained),algoritma youtube☠☠☠☠☠,22,2,False,False,False,False,True,False,True
369,UgxwnNRw_h5waAYDWVB4AaABAg,Kenapa Rupiah Melemah? (Explained),"Kalau seluruh rakyat Indonesia hobynya makan bakso semua,saya kira rupiah pasti akan kembali menguat.itulah masalah ...",164,22,False,False,False,False,True,False,True
381,UgzEJVnKPgy6novi5dZ4AaABAg,Kenapa Rupiah Melemah? (Explained),"Lucu banget, apa lagi pas masalah baksooooo hahahahahaha",56,8,False,False,False,False,True,False,True
399,Ugxfuz1n-5zEwdfsZdV4AaABAg,Kenapa Rupiah Melemah? (Explained),Lewat beranda lho ya😂😅😊😢😢😢😢😢,28,4,False,False,False,False,True,False,True
419,Ugz04PHVqRv23fK00-94AaABAg,Kenapa Rupiah Melemah? (Explained),2026.....seeeee,15,1,False,True,False,False,True,False,True


## 18. Ringkasan Kelayakan Dataset Sebelum Cleaning

Bagian ini menyusun ringkasan kelayakan dataset untuk menentukan apakah data sudah siap masuk ke tahap cleaning dan preprocessing NLP.

Kriteria yang digunakan bersifat praktis untuk audit awal:

- Kolom wajib tersedia.
- Jumlah baris memadai.
- Kolom `comment_id` tidak memiliki duplikasi tinggi.
- Kolom `text_original` tersedia dan tidak didominasi missing value.
- Tanggal komentar dapat dibaca.
- Proporsi noise masih dapat ditangani pada tahap cleaning.

In [34]:
# ============================================================
# Ringkasan Kelayakan Dataset Sebelum Masuk Tahap Cleaning
# ============================================================

# Status validasi kolom wajib
if len(missing_required_columns) == 0:
    column_status = "PASS"
    column_note = "Semua kolom wajib tersedia."
else:
    column_status = "FAIL"
    column_note = f"Masih ada kolom wajib yang belum tersedia: {missing_required_columns}"

# Status jumlah data
total_rows = df_raw.shape[0]

if total_rows >= 5000:
    row_status = "PASS"
    row_note = "Jumlah data sudah memenuhi target ideal minimal 5.000 komentar."
elif total_rows >= 3000:
    row_status = "WARNING"
    row_note = "Jumlah data cukup untuk audit dan eksplorasi awal, tetapi belum mencapai target ideal 5.000 komentar."
else:
    row_status = "WARNING"
    row_note = "Jumlah data masih relatif terbatas dan disarankan dilakukan crawling tambahan."

# Status duplikasi comment_id
duplicate_comment_id_count = 0

if "comment_id" in df_raw.columns:
    duplicate_comment_id_count = int(df_raw.duplicated(subset=["comment_id"]).sum())

if duplicate_comment_id_count == 0:
    duplicate_comment_id_status = "PASS"
    duplicate_comment_id_note = "Tidak ditemukan duplikasi berdasarkan comment_id."
else:
    duplicate_comment_id_status = "WARNING"
    duplicate_comment_id_note = f"Ditemukan {duplicate_comment_id_count} duplikasi berdasarkan comment_id."

# Status duplikasi text_original
duplicate_text_count = 0

if "text_original" in df_raw.columns:
    duplicate_text_count = int(df_raw.duplicated(subset=["text_original"]).sum())

if duplicate_text_count == 0:
    duplicate_text_status = "PASS"
    duplicate_text_note = "Tidak ditemukan duplikasi berdasarkan text_original."
else:
    duplicate_text_status = "WARNING"
    duplicate_text_note = f"Ditemukan {duplicate_text_count} duplikasi berdasarkan text_original."

# Status tanggal komentar
if "published_at_dt" in df_raw.columns:
    valid_date_count = int(df_raw["published_at_dt"].notna().sum())
elif "published_at_dt" in df_audit.columns:
    valid_date_count = int(df_audit["published_at_dt"].notna().sum())
else:
    valid_date_count = 0

if valid_date_count > 0:
    date_status = "PASS"
    date_note = "Kolom tanggal komentar dapat dikonversi dan dianalisis."
else:
    date_status = "FAIL"
    date_note = "Kolom tanggal komentar belum valid atau belum berhasil dikonversi."

# Status noise
noise_pct = 0.0

if "noise_summary_df" in globals() and not noise_summary_df.empty:
    if {"indicator", "percentage"}.issubset(noise_summary_df.columns):
        selected_noise = noise_summary_df.loc[
            noise_summary_df["indicator"] == "total_potential_noise",
            "percentage"
        ]
        
        if not selected_noise.empty:
            noise_pct = float(selected_noise.iloc[0])

if noise_pct <= 20:
    noise_status = "PASS"
    noise_note = f"Proporsi komentar berpotensi noise relatif terkendali, yaitu {noise_pct:.2f}%."
elif noise_pct <= 40:
    noise_status = "WARNING"
    noise_note = f"Proporsi komentar berpotensi noise cukup tinggi, yaitu {noise_pct:.2f}%."
else:
    noise_status = "WARNING"
    noise_note = f"Proporsi komentar berpotensi noise tinggi, yaitu {noise_pct:.2f}%, sehingga cleaning perlu dilakukan secara hati-hati."

# Ringkasan kelayakan dataset
dataset_readiness_summary_df = pd.DataFrame({
    "audit_aspect": [
        "Struktur kolom wajib",
        "Jumlah baris dataset",
        "Duplikasi comment_id",
        "Duplikasi text_original",
        "Validitas tanggal komentar",
        "Proporsi noise awal"
    ],
    "status": [
        column_status,
        row_status,
        duplicate_comment_id_status,
        duplicate_text_status,
        date_status,
        noise_status
    ],
    "note": [
        column_note,
        row_note,
        duplicate_comment_id_note,
        duplicate_text_note,
        date_note,
        noise_note
    ]
})

display(dataset_readiness_summary_df)

,audit_aspect,status,note
0,Struktur kolom wajib,PASS,Semua kolom wajib tersedia.
1,Jumlah baris dataset,WARNING,"Jumlah data cukup untuk audit dan eksplorasi awal, tetapi belum mencapai target ideal 5.000 komentar."
2,Duplikasi comment_id,PASS,Tidak ditemukan duplikasi berdasarkan comment_id.
3,Duplikasi text_original,WARNING,Ditemukan 43 duplikasi berdasarkan text_original.
4,Validitas tanggal komentar,FAIL,Kolom tanggal komentar belum valid atau belum berhasil dikonversi.
5,Proporsi noise awal,PASS,"Proporsi komentar berpotensi noise relatif terkendali, yaitu 5.69%."


## Interpretasi Ringkasan Kelayakan Dataset

Ringkasan kelayakan dataset menunjukkan apakah dataset raw komentar YouTube sudah cukup layak untuk dilanjutkan ke tahap cleaning dan preprocessing.

Aspek yang diperiksa meliputi kelengkapan kolom wajib, jumlah baris dataset, duplikasi berdasarkan `comment_id`, duplikasi berdasarkan `text_original`, validitas tanggal komentar, serta proporsi komentar yang berpotensi noise.

Status `PASS` menunjukkan aspek tersebut relatif aman untuk dilanjutkan, sedangkan status `WARNING` menunjukkan adanya catatan yang perlu diperhatikan pada tahap cleaning. Status `FAIL` menunjukkan aspek yang perlu diperbaiki sebelum analisis dilanjutkan.